# Bloom Filter — Probabilistic Membership at Scale

## 🧠 Mental Model

> **A Bloom filter is a bouncer with a perfect memory for "definitely NOT here"
> but occasional false confidence about "maybe here." No false negatives —
> if it says "no", trust it 100%. If it says "yes", verify with the DB.**

### The Core Trade-off

```
Bloom Filter answers: "Is X in the set?"

→ "NO"  → GUARANTEED correct   (zero false negatives — never misses a real member)
→ "YES" → PROBABLY correct     (bounded false positive rate — may claim non-members)

In exchange: uses a fraction of the memory a HashSet would use, and
NEVER stores the actual elements (privacy-preserving!).
```

### How It Works

```
BIT ARRAY of m bits (all start at 0)    k HASH FUNCTIONS

ADD "user:42":
  hash1("user:42") = 3  → set bit 3
  hash2("user:42") = 17 → set bit 17
  hash3("user:42") = 89 → set bit 89

CHECK "user:42": bits 3✓ 17✓ 89✓ → "PROBABLY in set" ✓
CHECK "user:99": bits 3✓ 17✓ 7✗  → "DEFINITELY NOT in set" ✓
CHECK "ghost":   bits 3✓ 17✓ 89✓ → "PROBABLY in set" ← FALSE POSITIVE!
                                      (bits set by other elements)
```

### Optimal Parameters

```
m (bits)  = -(n × ln(p)) / (ln 2)²    n = expected items, p = target FP rate
k (hashes)= (m/n) × ln 2

n=1M items, p=1%:   m = 9.58M bits (~1.2MB), k = 7 hashes
n=1M items, p=0.1%: m = 14.4M bits (~1.8MB), k = 10 hashes

HashSet for same 1M strings (~50 bytes each) = ~50MB
Bloom filter: ~1.2MB — 40× smaller at 1% FP rate
```

### 🌍 Where Bloom Filters are Used in Production

| System | What it filters | Why |
|---|---|---|
| Apache Cassandra | SSTable membership | Skip reading SSTables that DON'T contain the key |
| Apache HBase | HFile row existence | Avoid I/O for absent keys (eliminates 70-80% of reads) |
| Google Chrome | Safe Browsing | Check URLs against local filter before server query |
| Medium | Recommendations | "Have I shown this article to this user?" |
| Akamai CDN | One-hit-wonder filter | Don't cache content requested only once |
| Redis | `BF.ADD / BF.EXISTS` | Built-in via RedisBloom module |

### ⚠️ Gotchas

- **You cannot REMOVE elements** from a basic Bloom filter. Clearing bits breaks other elements.
  → Use **Counting Bloom Filter** (count instead of bits) if deletion is needed.
- **FP rate grows as the filter fills.** Size for your expected MAX load, not current.
- **Hash function quality matters.** Use independent functions (double-hashing is common).
  MD5/SHA1 are fine for non-adversarial inputs.
- **Cannot iterate or retrieve elements.** It is a membership oracle only.

---
## Implementation

In [ ]:
"""
04 — System Design: Bloom Filter (probabilistic membership at scale)
====================================================================

Runnable companion to PDF Book V "Answering 'have I seen this before?' cheaply".

A Bloom filter is a compact, probabilistic set. It answers "is X in the set?"
using a bit array and k hash functions. Its defining trade-off:

    * "NO"  is always correct   (no false negatives)
    * "YES" might be wrong      (a bounded false-positive rate)

In exchange it uses a fraction of the memory a real set would, and never stores
the elements themselves. Used to skip disk/DB/network lookups for keys that are
definitely absent: Cassandra/HBase SSTables, CDNs, "have we emailed this user?",
malicious-URL checks.

This file builds one from scratch and verifies: zero false negatives, and a
false-positive rate close to the theoretical prediction.
"""

from __future__ import annotations

import hashlib
import math

In [ ]:
class BloomFilter:
    def __init__(self, expected_items: int, false_positive_rate: float = 0.01):
        # Optimal bit-array size m and hash count k from the standard formulas.
        self._m = self._optimal_m(expected_items, false_positive_rate)
        self._k = self._optimal_k(self._m, expected_items)
        self._bits = bytearray((self._m + 7) // 8)
        self._n = 0

    @staticmethod
    def _optimal_m(n: int, p: float) -> int:
        return max(8, int(-(n * math.log(p)) / (math.log(2) ** 2)))

    @staticmethod
    def _optimal_k(m: int, n: int) -> int:
        return max(1, round((m / n) * math.log(2)))

    def _indexes(self, item: str):
        # Double hashing: derive k indexes from two base hashes (Kirsch-Mitzenmacher).
        h1 = int(hashlib.md5(item.encode()).hexdigest(), 16)
        h2 = int(hashlib.sha1(item.encode()).hexdigest(), 16)
        for i in range(self._k):
            yield (h1 + i * h2) % self._m

    def add(self, item: str) -> None:
        for idx in self._indexes(item):
            self._bits[idx // 8] |= (1 << (idx % 8))
        self._n += 1

    def __contains__(self, item: str) -> bool:
        return all(self._bits[idx // 8] & (1 << (idx % 8)) for idx in self._indexes(item))

In [ ]:
def demo() -> None:
    bf = BloomFilter(expected_items=10_000, false_positive_rate=0.01)
    present = [f"user:{i}" for i in range(10_000)]
    for item in present:
        bf.add(item)

    # No false negatives — everything added must report present.
    assert all(item in bf for item in present), "Bloom filters never have false negatives"
    print(f"   {len(present)} items added; zero false negatives (guaranteed)")

    # Measure the false-positive rate on 10k keys we never inserted.
    absent = [f"ghost:{i}" for i in range(10_000)]
    false_positives = sum(1 for item in absent if item in bf)
    fp_rate = false_positives / len(absent)
    assert fp_rate < 0.05, f"false-positive rate should be near 1%, got {fp_rate:.3f}"
    print(f"   false-positive rate on unseen keys: {fp_rate:.2%} (target ~1%)")

    # Memory: bits, not objects. Report the compression vs storing the strings.
    bloom_bytes = len(bf._bits)
    naive_bytes = sum(len(s) for s in present)   # rough lower bound for a real set
    print(f"   memory: {bloom_bytes:,} bytes vs ~{naive_bytes:,} bytes to store the keys "
          f"({naive_bytes / bloom_bytes:.0f}x smaller)")

In [ ]:
def main() -> None:
    print("=" * 70)
    print("SYSTEM DESIGN — bloom_filter.py")
    print("=" * 70)
    print("A probabilistic set: 'no' is certain, 'yes' is probable, memory is tiny:")
    demo()
    print("-" * 70)
    print("Lesson: use a Bloom filter to cheaply skip lookups for keys that are DEFINITELY absent (no false negatives).")
    print("All bloom_filter demos passed ✔")


if __name__ == "__main__":
    import sys
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()